# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neha-raniii/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/neha-raniii/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

feature_cols = ['impressions_90d', 'avg_position', 'ctr', 'word_count',
                 'days_since_last_update', 'search_volume', 'engagement_rate']
model_df = df.dropna(subset=feature_cols + ['client_id']).copy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(model_df, groups=model_df['client_id']))
train_df, test_df = model_df.iloc[train_idx], model_df.iloc[test_idx]

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(train_df[feature_cols], train_df['is_declining_label'])

print(f"{len(df):,} rows loaded, model trained on {len(train_df):,} rows")

30,000 rows loaded, model trained on 14,160 rows


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Ranked queue: every eligible page scored by the trained model's decline probability, sorted highest to lowest, with a reason code and suggested action a reviewer can trust and inspect.

In [16]:
scored = model_df.copy()
scored['model_score'] = model.predict_proba(scored[feature_cols])[:, 1]

def reason_and_action(row):
    if row['model_score'] >= 0.65 and row['days_since_last_update'] >= 180:
        return 'high_risk_stale_visible', 'review_and_refresh'
    elif row['model_score'] >= 0.65:
        return 'high_risk_decline', 'review_content'
    elif row['ctr'] < row['impressions_90d'] * 0 + 0.2 and row['avg_position'] <= 20 and row['impressions_90d'] >= 500:
        return 'ctr_below_expected', 'review_ctr_and_metadata'
    else:
        return 'monitor', 'monitor'

scored[['reason_code', 'action']] = scored.apply(lambda r: pd.Series(reason_and_action(r)), axis=1)

queue = scored.sort_values('model_score', ascending=False)[
    ['content_id', 'client_id', 'model_score', 'impressions_90d', 'avg_position',
     'ctr', 'days_since_last_update', 'reason_code', 'action']
].reset_index(drop=True)

print(f"Queue built: {len(queue):,} pages scored")
print(queue['reason_code'].value_counts())
queue.head(10)


Queue built: 20,018 pages scored
reason_code
high_risk_decline          11294
monitor                     7562
ctr_below_expected          1115
high_risk_stale_visible       47
Name: count, dtype: int64


,content_id,client_id,model_score,impressions_90d,avg_position,ctr,days_since_last_update,reason_code,action
0,content_a16cc7d0e1da,client_3fdba35f04,1.0,235,4.0,0.00,104,high_risk_decline,review_content
1,content_df5cd60fe5be,client_6208ef0f77,1.0,247,11.2,0.00,104,high_risk_decline,review_content
2,content_4545e0f37871,client_3fdba35f04,1.0,157,5.2,0.00,104,high_risk_decline,review_content
3,content_2f03e29a8c2a,client_3fdba35f04,1.0,251,22.4,0.00,104,high_risk_decline,review_content
4,content_1c38d62e358d,client_6208ef0f77,1.0,473,7.6,0.00,104,high_risk_decline,review_content
5,content_bd6b3cb3b2d5,client_6208ef0f77,1.0,179,9.7,0.00,104,high_risk_decline,review_content
6,content_9d4e9f7b6e58,client_3fdba35f04,1.0,273,5.4,0.00,104,high_risk_decline,review_content
7,content_0c72c3e22858,client_3fdba35f04,1.0,1845,28.7,0.00,104,high_risk_decline,review_content
8,content_9c8f9dc24e85,client_3fdba35f04,1.0,2229,30.9,0.09,104,high_risk_decline,review_content
9,content_99ba59484a1d,client_6208ef0f77,1.0,1589,5.5,0.06,104,high_risk_decline,review_content


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended use: This ranked queue is for a content review team with limited weekly capacity, to decide which pages to check first. It supports human decision-making - it is not an automated action system, and no page should be edited, unpublished, or removed based on the score alone.

Where it stops being valid:
- This queue is built from the 30,000-row starter dataset, not the full 79M-row warehouse - it has not been validated at that scale.
- The label (is_declining_label) is a current-window bucket, not a future outcome, so "high_risk_decline" means "currently trending down," not "guaranteed to keep declining."
- The model was trained and evaluated with a client-holdout split (Precision@50 = 0.74), meaning roughly 1 in 4 of the top-ranked pages will still be a false


In [17]:
# Intended use and limits are conceptual; no additional computation needed for this section.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

What a human must check before acting on any flagged page:
- Verify the numbers aren't a tracking artifact (e.g. a CTR of exactly 0.00 with real impressions, as seen in several rows above - this could be a genuine problem or a measurement glitch).
- Check whether a "declining" page is actually a case of consolidation (a sibling page absorbed its traffic) rather than a real drop - the model has no way to detect this.
- Confirm the page still matches current business priorities before investing review time in it.

No-go list - what should NEVER be automated from this queue:
- Never auto-delete, auto-unpublish, or auto-redirect a page based on model_score alone.
- Never present "high_risk_decline" to a client or stakeholder as a guarantee - it is a probability from one model on one split, not a certainty.
- Never skip human review for pages with very high impressions_90d - these are exactly the pages where a wrong call is most costly, so they deserve more scrutiny, not less.
- Never treat the reason_code as a root-cause diagnosis - it flags where to look, not what is definitely wrong.

In [18]:
# Review rules and no-go list are policy statements, not computations.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Signals that would tell me these recommendations have gone stale and need a retrain or review:

1. Precision@50 drop on fresh data - if a new batch of pages is scored and manually checked, and the actual hit rate falls meaningfully below the validated 0.74, the model's signal has likely shifted.
2. Feature distribution shift - if the average impressions_90d, avg_position, or ctr across new pages looks very different from the training data's distribution, the model is scoring pages unlike anything it learned from.
3. Reason code imbalance shift - if high_risk_decline suddenly jumps from ~56% of the queue (current) to a very different share, something upstream (tracking, business conditions, or genuine market shift) has likely changed and the queue should be re-examined before trusting it.
4. Time elapsed - since this is a single snapshot, a hard rule of re-scoring at least every 90 days (matching the dataset's own impressions_90d window) keeps the queue from running on outdated signal.
5. New client onboarding - since validation was client-holdout, a genuinely new client's pages should be treated with lower initial confidence until enough of their own history accumulates.

In [19]:
# Monitoring triggers are policy/process statements, not computations.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Exporting the ranked queue and a summary of reason codes to work/outputs/, so the capstone paper can reference these files directly.

In [20]:

import os
os.makedirs('work/outputs', exist_ok=True)

# Full ranked queue
queue.to_csv('work/outputs/action_playbook_queue.csv', index=False)

# Summary stats (safe to commit - this is a JSON, not raw data)
import json
summary = {
    'total_pages_scored': len(queue),
    'reason_code_counts': queue['reason_code'].value_counts().to_dict(),
    'model_precision_at_50': 0.74,
    'baseline_precision_at_50': 0.56,
    'validation_method': 'client_holdout_group_split',
    'dataset': 'starter_30k_content_refresh_anonymized'
}
with open('work/outputs/action_playbook_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("Exported:")
print("- work/outputs/action_playbook_queue.csv")
print("- work/outputs/action_playbook_summary.json")
print("\nSummary:", json.dumps(summary, indent=2))

Exported:
- work/outputs/action_playbook_queue.csv
- work/outputs/action_playbook_summary.json

Summary: {
  "total_pages_scored": 20018,
  "reason_code_counts": {
    "high_risk_decline": 11294,
    "monitor": 7562,
    "ctr_below_expected": 1115,
    "high_risk_stale_visible": 47
  },
  "model_precision_at_50": 0.74,
  "baseline_precision_at_50": 0.56,
  "validation_method": "client_holdout_group_split",
  "dataset": "starter_30k_content_refresh_anonymized"
}


In [21]:
from google.colab import files
files.download('work/outputs/action_playbook_summary.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.